In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q transformers==4.45.0 peft==0.13.0 trl==0.11.0 \
             datasets==2.19.0 accelerate==0.34.0 sentencepiece scipy lm-eval

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 75.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 105.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, json, random
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets
from lm_eval.models.huggingface import HFLM

MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
OUTPUT_DIR  = "/kaggle/working/tinyllama-medical-lora"
MERGED_DIR  = "/kaggle/working/tinyllama-medical-merged"
MAX_SEQ_LEN = 512
BATCH_SIZE  = 2
GRAD_ACCUM  = 16
EPOCHS     = 1
LR         = 1e-4
LORA_R     = 256
LORA_ALPHA = 256
LORA_DROPOUT= 0.05

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 49,020,928 || all params: 1,149,069,312 || trainable%: 4.2661


In [6]:
def fmt(text):
    return str(text).strip()

def format_medqa(ex):
    opts = ex['options']
    choices = "\n".join([f"({k}) {v}" for k, v in opts.items()])
    
    prompt = (
        f"Question: {fmt(ex['question'])}\n\n"
        f"{choices}\n\n"
        f"Answer: ("
    )
    answer = f"{fmt(ex['answer_idx'])})"
    
    return {
        "prompt": prompt,
        "answer": answer,
        "text": prompt + answer
    }

medqa_train = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
medqa_val   = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")

medqa_train = medqa_train.map(format_medqa, remove_columns=medqa_train.column_names)
medqa_val   = medqa_val.map(format_medqa,   remove_columns=medqa_val.column_names)

print(f"Train: {len(medqa_train):,}")
print(f"Val  : {len(medqa_val):,}")
print(f"\nSample:\n{medqa_train[0]['text']}")

Train: 10,178
Val  : 1,273

Sample:
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

(A) Ampicillin
(B) Ceftriaxone
(C) Doxycycline
(D) Nitrofurantoin

Answer: (D)


In [7]:
from transformers import Trainer

def make_collator(tokenizer, max_len):
    def collate(batch):
        full_texts = [b["text"] for b in batch]
        
        enc = tokenizer(full_texts, truncation=True, max_length=max_len,
                        padding="max_length", return_tensors="pt")
        
        labels = enc["input_ids"].clone()

        for i, text in enumerate(full_texts):
            split_marker = "Answer: ("
            prompt_part  = text[:text.rfind(split_marker) + len(split_marker)]
            prompt_ids   = tokenizer(prompt_part, add_special_tokens=False)["input_ids"]
            prompt_len   = len(prompt_ids)
            labels[i, :prompt_len] = -100

        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels
        return enc
    return collate

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    group_by_length=False,   # must be False — dataset has no input_ids column
    optim="adamw_torch",
    weight_decay=0.01,
    remove_unused_columns=False,
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=make_collator(tokenizer, MAX_SEQ_LEN),
    train_dataset=medqa_train,
    eval_dataset=medqa_val,
    args=training_args,
)

trainer.train()
print("Done!")

Step,Training Loss,Validation Loss
100,0.465900,0.463777
200,0.462600,0.466038
300,0.462700,0.462245


Done!


In [8]:
from lm_eval.models.huggingface import HFLM
import lm_eval
lm_model_ft = HFLM(
    pretrained=trainer.model,
    tokenizer=tokenizer,
    batch_size=8,
)

results_after = lm_eval.simple_evaluate(
    model=lm_model_ft,
    tasks=["medqa_4options", "pubmedqa"],
)

medqa_after  = results_after["results"]["medqa_4options"]["acc,none"] * 100
pubmed_after = results_after["results"]["pubmedqa"]["acc,none"] * 100

print(f"  medqa_4options : {medqa_after:.2f}  (baseline: 24.74)")
print(f"  pubmedqa       : {pubmed_after:.2f}  (baseline: 60.80)")

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1272 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for bigbio/pubmed_qa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/pubmed_qa
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Running loglikelihood requests: 100%|██████████| 6592/6592 [09:41<00:00, 11.33it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


  medqa_4options : 25.45  (baseline: 24.74)
  pubmedqa       : 60.40  (baseline: 60.80)
